In [42]:
# ============================================================
# MatricMath Intelligence
# Notebook 02: Document Collection & Extraction
# ============================================================
# Purpose:
# Collect ONLY documents verified in Notebook 01.
# Resolve official PDF URLs, download, validate, and update
# the control register.
#
# Rule:
# Never download unless collection_status == "verified"
# ============================================================

print("Notebook 02 – Document Collection & Extraction")

Notebook 02 – Document Collection & Extraction


In [43]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin
from html.parser import HTMLParser
import re
import time

import pandas as pd
import requests

In [44]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
META_DIR = DATA_DIR / "metadata"

EXAMS_DIR = RAW_DIR / "exams"
MEMOS_DIR = RAW_DIR / "memos"
DIAG_DIR = RAW_DIR / "diagnostic_reports"
CAPS_DIR = RAW_DIR / "caps"

for d in [EXAMS_DIR, MEMOS_DIR, DIAG_DIR, CAPS_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REGISTER_PATH = META_DIR / "exam_document_register.csv"

print("Project root:", PROJECT_ROOT)
print("Register    :", REGISTER_PATH)

Project root: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence
Register    : c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\exam_document_register.csv


In [45]:
if not REGISTER_PATH.exists():
    raise FileNotFoundError(
        f"Register not found: {REGISTER_PATH}\nRun Notebook 01 first."
    )

register_df = pd.read_csv(REGISTER_PATH)

# Ensure text columns are object dtype
for col in ["source_url", "file_name", "file_path", "notes", "language"]:
    if col in register_df.columns:
        register_df[col] = register_df[col].astype("object")
        register_df[col] = register_df[col].where(register_df[col].notna(), "")

required_columns = [
    "document_id", "year", "document_type", "source", "source_tier",
    "source_url", "collection_status", "file_name", "file_path", "notes"
]

missing = [c for c in required_columns if c not in register_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Register loaded: {len(register_df)} rows")
print(register_df["collection_status"].value_counts())

Register loaded: 206 rows
collection_status
queued      192
verified     14
Name: count, dtype: int64


In [46]:
verified_df = register_df[
    register_df["collection_status"] == "verified"
].copy()

print("=" * 60)
print("COLLECTION QUEUE (verified only)")
print("=" * 60)
print(f"Eligible documents: {len(verified_df)}")

display(
    verified_df[
        ["document_id", "year", "exam_session", "paper",
         "document_type", "source_tier", "source_url"]
    ].sort_values(["document_type", "year", "paper"])
)

# Safety: Tier 1 only for first collection wave
non_tier1 = verified_df[verified_df["source_tier"] != 1]
if len(non_tier1) > 0:
    print("WARNING: non-Tier-1 documents present")
    display(non_tier1[["document_id", "source", "source_tier"]])
else:
    print("All queue items are Tier 1.")

COLLECTION QUEUE (verified only)
Eligible documents: 14


,document_id,year,exam_session,paper,document_type,source_tier,source_url
204,caps_mathematics_gr10_12,2011,NaN,NaN,caps,1,https://www.education.gov.za/Portals/0/CD/Nati...
144,2023_nov_p1_exam_maths,2023,Nov,P1,exam,1,https://www.education.gov.za/2023NSCNovemberpa...
146,2023_nov_p2_exam_maths,2023,Nov,P2,exam,1,https://www.education.gov.za/2023NSCNovemberpa...
160,2024_nov_p1_exam_maths,2024,Nov,P1,exam,1,https://www.education.gov.za/2024NSCNovemberpa...
162,2024_nov_p2_exam_maths,2024,Nov,P2,exam,1,https://www.education.gov.za/2024NSCNovemberpa...
176,2025_nov_p1_exam_maths,2025,Nov,P1,exam,1,https://www.education.gov.za/Curriculum/Nation...
178,2025_nov_p2_exam_maths,2025,Nov,P2,exam,1,https://www.education.gov.za/Curriculum/Nation...
205,grade12_mathematics_exam_guideline_2021,2021,NaN,NaN,guideline,1,https://www.education.gov.za/Portals/0/CD/2021...
145,2023_nov_p1_memo_maths,2023,Nov,P1,memo,1,https://www.education.gov.za/2023NSCNovemberpa...
147,2023_nov_p2_memo_maths,2023,Nov,P2,memo,1,https://www.education.gov.za/2023NSCNovemberpa...


All queue items are Tier 1.


In [47]:
def clean_text(value) -> str:
    if pd.isna(value):
        return ""
    value = str(value).strip()
    value = re.sub(r"[^A-Za-z0-9_-]+", "_", value)
    return value.strip("_")


def generate_filename(row) -> str:
    year = clean_text(row["year"])
    session = clean_text(row.get("exam_session", ""))
    paper = clean_text(row.get("paper", ""))
    doc_type = clean_text(row["document_type"])

    parts = ["NSC", "Mathematics", year]
    if session and session.upper() not in ["NA", "NAN", ""]:
        parts.append(session)
    if paper and paper.upper() not in ["NA", "NAN", ""]:
        parts.append(paper)
    parts.append(doc_type)
    return "_".join(parts) + ".pdf"


def get_storage_directory(document_type: str) -> Path:
    t = str(document_type).lower().strip()
    if t == "exam":
        return EXAMS_DIR
    if t == "memo":
        return MEMOS_DIR
    if t == "diagnostic":
        return DIAG_DIR
    if t in ["caps", "guideline", "exam_guideline"]:
        return CAPS_DIR
    return RAW_DIR


# Filename preview
for _, row in verified_df.head(8).iterrows():
    print(row["document_id"], "→", generate_filename(row))

2023_nov_p1_exam_maths → NSC_Mathematics_2023_Nov_P1_exam.pdf
2023_nov_p1_memo_maths → NSC_Mathematics_2023_Nov_P1_memo.pdf
2023_nov_p2_exam_maths → NSC_Mathematics_2023_Nov_P2_exam.pdf
2023_nov_p2_memo_maths → NSC_Mathematics_2023_Nov_P2_memo.pdf
2024_nov_p1_exam_maths → NSC_Mathematics_2024_Nov_P1_exam.pdf
2024_nov_p1_memo_maths → NSC_Mathematics_2024_Nov_P1_memo.pdf
2024_nov_p2_exam_maths → NSC_Mathematics_2024_Nov_P2_exam.pdf
2024_nov_p2_memo_maths → NSC_Mathematics_2024_Nov_P2_memo.pdf


In [48]:
class PDFLinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.pdf_links = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() != "a":
            return
        href = dict(attrs).get("href")
        if href and ".pdf" in href.lower():
            self.pdf_links.append(href)


REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}


def is_direct_pdf_url(url: str) -> bool:
    return isinstance(url, str) and url.lower().endswith(".pdf")


def get_pdf_links(page_url: str):
    """Return absolute PDF links from a webpage."""
    response = requests.get(page_url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()

    parser = PDFLinkParser()
    parser.feed(response.text)

    return [urljoin(page_url, link) for link in parser.pdf_links]


def normalise_for_matching(text: str) -> str:
    text = str(text).lower().replace("-", " ").replace("_", " ")
    return re.sub(r"\s+", " ", text)


def score_pdf_link(link: str, row) -> int:
    """Higher score = better match to target document."""
    text = normalise_for_matching(link)
    score = 0

    if "mathematics" in text or " maths " in f" {text} ":
        score += 5

    year = str(row["year"])
    if year in text:
        score += 3

    paper = str(row.get("paper", "")).lower()
    if paper == "p1" and ("paper 1" in text or "paper1" in text or " p1" in text):
        score += 5
    if paper == "p2" and ("paper 2" in text or "paper2" in text or " p2" in text):
        score += 5

    doc_type = str(row["document_type"]).lower()
    if doc_type == "memo":
        if "memo" in text or "marking" in text:
            score += 5
        if "paper" in text and "memo" not in text:
            score -= 2
    elif doc_type == "exam":
        if "memo" in text or "marking guideline" in text:
            score -= 4
        else:
            score += 2

    # Language preference (English first for this wave)
    if "english" in text or "eng" in text:
        score += 1
    if "afrikaans" in text and "english" not in text:
        score -= 1

    return score

In [49]:
# ============================================================
# INSPECT PDF CANDIDATES (DO NOT DOWNLOAD YET)
# ============================================================

source_pages = (
    verified_df["source_url"]
    .dropna()
    .astype(str)
    .loc[lambda s: s.str.len() > 0]
    .unique()
)

for page_url in source_pages:
    print("=" * 80)
    print(page_url)
    print("=" * 80)

    try:
        if is_direct_pdf_url(page_url):
            print("Direct PDF URL (no page scrape needed)")
            print(page_url)
        else:
            links = get_pdf_links(page_url)
            print(f"PDF links found: {len(links)}")
            for link in links[:25]:
                print(link)
    except Exception as e:
        print("ERROR:", e)

    print()

https://www.education.gov.za/2023NSCNovemberpastpapers.aspx


PDF links found: 0

https://www.education.gov.za/2024NSCNovemberpastpapers.aspx
PDF links found: 0

https://www.education.gov.za/Curriculum/NationalSeniorCertificate%28NSC%29Examinations/2025NovemberExamPapers.aspx
PDF links found: 0

https://www.education.gov.za/Portals/0/CD/National%20Curriculum%20Statements%20and%20Vocational/CAPS%20FET%20_%20MATHEMATICS%20_%20GR%2010-12%20_%20Web_1133.pdf
Direct PDF URL (no page scrape needed)
https://www.education.gov.za/Portals/0/CD/National%20Curriculum%20Statements%20and%20Vocational/CAPS%20FET%20_%20MATHEMATICS%20_%20GR%2010-12%20_%20Web_1133.pdf

https://www.education.gov.za/Portals/0/CD/2021%20Exam%20Guidelines/Mathematics%20GR%2012%20Exam%20Guidelines%202021%20Eng.pdf
Direct PDF URL (no page scrape needed)
https://www.education.gov.za/Portals/0/CD/2021%20Exam%20Guidelines/Mathematics%20GR%2012%20Exam%20Guidelines%202021%20Eng.pdf



In [50]:
# ============================================================
# PDF RESOLUTION
# - Prefer manual direct PDF URLs for DBE exam/memo pages
# - Use source_url directly if it is already a PDF
# - Fallback to page scrape only if needed
# ============================================================

from urllib.parse import urljoin, unquote
from html.parser import HTMLParser
import re

class PDFLinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.pdf_links = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() != "a":
            return
        href = dict(attrs).get("href")
        if href and ".pdf" in href.lower():
            self.pdf_links.append(href)


SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
})

_pdf_link_cache = {}

def is_direct_pdf_url(url: str) -> bool:
    if not isinstance(url, str) or not url.strip():
        return False
    return url.lower().split("?")[0].endswith(".pdf")


def get_pdf_links(page_url: str) -> list:
    if page_url in _pdf_link_cache:
        return _pdf_link_cache[page_url]

    response = SESSION.get(page_url, timeout=30)
    response.raise_for_status()

    parser = PDFLinkParser()
    parser.feed(response.text)

    links = []
    seen = set()
    for link in parser.pdf_links:
        abs_link = urljoin(page_url, link)
        if abs_link not in seen:
            seen.add(abs_link)
            links.append(abs_link)

    _pdf_link_cache[page_url] = links
    return links


# ------------------------------------------------------------
# MANUAL DIRECT PDF MAP
# Fill these from browser: right-click Mathematics download link → Copy link address
# Use your exact document_id values from the register.
# ------------------------------------------------------------
MANUAL_PDF_MAP = {
    # 2025 November
    "2025_nov_p1_exam_maths": "https://www.education.gov.za/LinkClick.aspx?fileticket=JM4biRg1OIk%3d&tabid=5742&portalid=0&mid=14845",
    "2025_nov_p1_memo_maths": "https://www.education.gov.za/LinkClick.aspx?fileticket=lMX4KlIrUCs%3d&tabid=5742&portalid=0&mid=14845",
    "2025_nov_p2_exam_maths": "https://www.education.gov.za/LinkClick.aspx?fileticket=8t-92qfBEV0%3d&tabid=5742&portalid=0&mid=14845",
    "2025_nov_p2_memo_maths": "https://www.education.gov.za/LinkClick.aspx?fileticket=JM4biRg1OIk%3d&tabid=5742&portalid=0&mid=14845",

    # 2024 November
    "2024_nov_p1_exam_maths": "https://www.education.gov.za/Portals/0/CD/2024%20November%20past%20papers/Mathematics%20P1%20Nov%202024%20Eng.pdf",
    "2024_nov_p1_memo_maths": "https://www.education.gov.za/LinkClick.aspx?fileticket=D_T4clPBpkk%3d&tabid=5193&portalid=0&mid=13724",
    "2024_nov_p2_exam_maths": "https://www.education.gov.za/Portals/0/CD/2024%20November%20past%20papers/Mathematics%20P2%20Nov%202024%20Eng.pdf",
    "2024_nov_p2_memo_maths": "https://www.education.gov.za/LinkClick.aspx?fileticket=0DIM92_2Vu8%3d&tabid=5193&portalid=0&mid=13724",

    # 2023 November
    "2023_nov_p1_exam_maths": "https://www.education.gov.za/LinkClick.aspx?fileticket=M_7mZq2zE5o%3d&tabid=4682&portalid=0&mid=12681",
    "2023_nov_p1_memo_maths": "https://www.education.gov.za/LinkClick.aspx?fileticket=ViEtlf4659c%3d&tabid=4682&portalid=0&mid=12681",
    "2023_nov_p2_exam_maths": "https://www.education.gov.za/LinkClick.aspx?fileticket=Zoios-rCurI%3d&tabid=4682&portalid=0&mid=12681",
    "2023_nov_p2_memo_maths": "https://www.education.gov.za/LinkClick.aspx?fileticket=0RJSBcYBmhA%3d&tabid=4682&portalid=0&mid=12681",
}


def normalise(text: str) -> str:
    text = unquote(str(text).lower())
    text = text.replace("-", " ").replace("_", " ")
    return re.sub(r"\s+", " ", text)


def score_pdf_link(link: str, row) -> int:
    text = normalise(link)
    score = 0

    if "mathematics" in text:
        score += 8
    elif "maths" in text or re.search(r"\bmath\b", text):
        score += 5

    if "technical mathematics" in text or "mathematical literacy" in text:
        score -= 12

    year = str(row["year"])
    if year in text:
        score += 4

    paper = str(row.get("paper", "")).lower()
    if paper == "p1":
        if "paper 1" in text or "paper1" in text or re.search(r"\bp1\b", text):
            score += 6
        if "paper 2" in text or "paper2" in text or re.search(r"\bp2\b", text):
            score -= 4
    elif paper == "p2":
        if "paper 2" in text or "paper2" in text or re.search(r"\bp2\b", text):
            score += 6
        if "paper 1" in text or "paper1" in text or re.search(r"\bp1\b", text):
            score -= 4

    doc_type = str(row["document_type"]).lower()
    if doc_type == "memo":
        if any(k in text for k in ["memo", "marking guideline", "marking"]):
            score += 7
        if "question paper" in text and "memo" not in text:
            score -= 3
    elif doc_type == "exam":
        if "memo" in text or "marking guideline" in text:
            score -= 6
        if "question paper" in text or "paper" in text:
            score += 3

    if "english" in text or re.search(r"\beng\b", text):
        score += 2
    if "afrikaans" in text or re.search(r"\bafr\b", text):
        score -= 1

    return score


def resolve_pdf_url(row) -> str:
    document_id = str(row["document_id"])

    # 1) Manual map first
    manual = MANUAL_PDF_MAP.get(document_id, "").strip()
    if manual:
        return manual

    source_url = str(row.get("source_url", "")).strip()
    if not source_url:
        raise ValueError("Missing source_url and no manual PDF map entry")

    # 2) Direct PDF already in register
    if is_direct_pdf_url(source_url):
        return source_url

    # 3) Scrape landing page
    links = get_pdf_links(source_url)
    if not links:
        raise ValueError(
            "No PDF links found on source page. "
            "Add a direct URL to MANUAL_PDF_MAP for this document_id."
        )

    scored = sorted(
        ((score_pdf_link(link, row), link) for link in links),
        key=lambda x: x[0],
        reverse=True
    )
    best_score, best_link = scored[0]
    if best_score < 8:
        top = "\n".join([f"  [{s}] {u}" for s, u in scored[:5]])
        raise ValueError(
            f"No confident PDF match (best_score={best_score}). "
            f"Add manual URL. Top candidates:\n{top}"
        )
    return best_link


print("PDF resolution helpers ready")
print("Manual map entries:", len(MANUAL_PDF_MAP))
print("Filled manual URLs:", sum(1 for v in MANUAL_PDF_MAP.values() if str(v).strip()))

PDF resolution helpers ready
Manual map entries: 12
Filled manual URLs: 12


In [51]:
need_manual = verified_df[
    verified_df["document_type"].isin(["exam", "memo"])
][["document_id", "year", "paper", "document_type", "source_url"]].copy()

need_manual["manual_url"] = need_manual["document_id"].map(MANUAL_PDF_MAP)
need_manual["manual_filled"] = need_manual["manual_url"].fillna("").astype(str).str.strip().ne("")

display(need_manual.sort_values(["year", "paper", "document_type"]))
print("Still missing manual URLs:", (~need_manual["manual_filled"]).sum())

,document_id,year,paper,document_type,source_url,manual_url,manual_filled
144,2023_nov_p1_exam_maths,2023,P1,exam,https://www.education.gov.za/2023NSCNovemberpa...,https://www.education.gov.za/LinkClick.aspx?fi...,True
145,2023_nov_p1_memo_maths,2023,P1,memo,https://www.education.gov.za/2023NSCNovemberpa...,https://www.education.gov.za/LinkClick.aspx?fi...,True
146,2023_nov_p2_exam_maths,2023,P2,exam,https://www.education.gov.za/2023NSCNovemberpa...,https://www.education.gov.za/LinkClick.aspx?fi...,True
147,2023_nov_p2_memo_maths,2023,P2,memo,https://www.education.gov.za/2023NSCNovemberpa...,https://www.education.gov.za/LinkClick.aspx?fi...,True
160,2024_nov_p1_exam_maths,2024,P1,exam,https://www.education.gov.za/2024NSCNovemberpa...,https://www.education.gov.za/Portals/0/CD/2024...,True
161,2024_nov_p1_memo_maths,2024,P1,memo,https://www.education.gov.za/2024NSCNovemberpa...,https://www.education.gov.za/LinkClick.aspx?fi...,True
162,2024_nov_p2_exam_maths,2024,P2,exam,https://www.education.gov.za/2024NSCNovemberpa...,https://www.education.gov.za/Portals/0/CD/2024...,True
163,2024_nov_p2_memo_maths,2024,P2,memo,https://www.education.gov.za/2024NSCNovemberpa...,https://www.education.gov.za/LinkClick.aspx?fi...,True
176,2025_nov_p1_exam_maths,2025,P1,exam,https://www.education.gov.za/Curriculum/Nation...,https://www.education.gov.za/LinkClick.aspx?fi...,True
177,2025_nov_p1_memo_maths,2025,P1,memo,https://www.education.gov.za/Curriculum/Nation...,https://www.education.gov.za/LinkClick.aspx?fi...,True


Still missing manual URLs: 0


In [52]:
preview_rows = []

for _, row in verified_df.iterrows():
    try:
        pdf_url = resolve_pdf_url(row)
        preview_rows.append({
            "document_id": row["document_id"],
            "document_type": row["document_type"],
            "year": row["year"],
            "paper": row.get("paper", ""),
            "matched_pdf": pdf_url,
            "status": "ok"
        })
    except Exception as e:
        preview_rows.append({
            "document_id": row["document_id"],
            "document_type": row["document_type"],
            "year": row["year"],
            "paper": row.get("paper", ""),
            "matched_pdf": "",
            "status": f"FAIL: {e}"
        })

preview_df = pd.DataFrame(preview_rows)
display(preview_df)

print("OK:", (preview_df["status"] == "ok").sum())
print("FAIL:", (preview_df["status"] != "ok").sum())

,document_id,document_type,year,paper,matched_pdf,status
0,2023_nov_p1_exam_maths,exam,2023,P1,https://www.education.gov.za/LinkClick.aspx?fi...,ok
1,2023_nov_p1_memo_maths,memo,2023,P1,https://www.education.gov.za/LinkClick.aspx?fi...,ok
2,2023_nov_p2_exam_maths,exam,2023,P2,https://www.education.gov.za/LinkClick.aspx?fi...,ok
3,2023_nov_p2_memo_maths,memo,2023,P2,https://www.education.gov.za/LinkClick.aspx?fi...,ok
4,2024_nov_p1_exam_maths,exam,2024,P1,https://www.education.gov.za/Portals/0/CD/2024...,ok
5,2024_nov_p1_memo_maths,memo,2024,P1,https://www.education.gov.za/LinkClick.aspx?fi...,ok
6,2024_nov_p2_exam_maths,exam,2024,P2,https://www.education.gov.za/Portals/0/CD/2024...,ok
7,2024_nov_p2_memo_maths,memo,2024,P2,https://www.education.gov.za/LinkClick.aspx?fi...,ok
8,2025_nov_p1_exam_maths,exam,2025,P1,https://www.education.gov.za/LinkClick.aspx?fi...,ok
9,2025_nov_p1_memo_maths,memo,2025,P1,https://www.education.gov.za/LinkClick.aspx?fi...,ok


OK: 14
FAIL: 0


In [53]:
def download_pdf(pdf_url: str, output_path: Path) -> None:
    response = SESSION.get(pdf_url, timeout=90)
    response.raise_for_status()

    if not response.content.startswith(b"%PDF"):
        raise ValueError("Downloaded content is not a PDF")
    if len(response.content) < 1000:
        raise ValueError("Downloaded PDF is suspiciously small")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "wb") as f:
        f.write(response.content)

In [54]:
download_log = []

for _, row in verified_df.iterrows():
    document_id = row["document_id"]
    try:
        pdf_url = resolve_pdf_url(row)
        filename = generate_filename(row)
        output_dir = get_storage_directory(row["document_type"])
        output_path = output_dir / filename

        print(f"\nDownloading: {document_id}")
        print(f"  URL : {pdf_url}")
        print(f"  File: {output_path}")

        download_pdf(pdf_url, output_path)

        idx = register_df.index[register_df["document_id"] == document_id]
        register_df.loc[idx, "file_name"] = filename
        register_df.loc[idx, "file_path"] = str(output_path.relative_to(PROJECT_ROOT))
        register_df.loc[idx, "collection_status"] = "downloaded"

        existing_notes = str(row.get("notes", "") or "")
        note_add = f"PDF URL: {pdf_url}"
        register_df.loc[idx, "notes"] = (
            f"{existing_notes} | {note_add}".strip(" |") if existing_notes else note_add
        )

        download_log.append({
            "document_id": document_id,
            "status": "downloaded",
            "pdf_url": pdf_url,
            "file_path": str(output_path)
        })
        time.sleep(1.2)

    except Exception as e:
        print(f"FAILED: {document_id}")
        print(f"  Reason: {e}")
        download_log.append({
            "document_id": document_id,
            "status": "failed",
            "reason": str(e)
        })

download_log_df = pd.DataFrame(download_log)
display(download_log_df)
print("Successful:", (download_log_df["status"] == "downloaded").sum())
print("Failed:", (download_log_df["status"] == "failed").sum())


Downloading: 2023_nov_p1_exam_maths
  URL : https://www.education.gov.za/LinkClick.aspx?fileticket=M_7mZq2zE5o%3d&tabid=4682&portalid=0&mid=12681
  File: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\exams\NSC_Mathematics_2023_Nov_P1_exam.pdf



Downloading: 2023_nov_p1_memo_maths
  URL : https://www.education.gov.za/LinkClick.aspx?fileticket=ViEtlf4659c%3d&tabid=4682&portalid=0&mid=12681
  File: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\memos\NSC_Mathematics_2023_Nov_P1_memo.pdf

Downloading: 2023_nov_p2_exam_maths
  URL : https://www.education.gov.za/LinkClick.aspx?fileticket=Zoios-rCurI%3d&tabid=4682&portalid=0&mid=12681
  File: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\exams\NSC_Mathematics_2023_Nov_P2_exam.pdf

Downloading: 2023_nov_p2_memo_maths
  URL : https://www.education.gov.za/LinkClick.aspx?fileticket=0RJSBcYBmhA%3d&tabid=4682&portalid=0&mid=12681
  File: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\memos\NSC_Mathematics_2023_Nov_P2_memo.pdf

Downloading: 2024_nov_p1_exam_maths
  URL : https://www.education.gov.za/Portals/0/CD/2024%20November%20past%20papers/Mathematics%20P1%20Nov%202024%20Eng.pdf
  File: c:\Users\Administrator\Des

,document_id,status,pdf_url,file_path
0,2023_nov_p1_exam_maths,downloaded,https://www.education.gov.za/LinkClick.aspx?fi...,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
1,2023_nov_p1_memo_maths,downloaded,https://www.education.gov.za/LinkClick.aspx?fi...,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
2,2023_nov_p2_exam_maths,downloaded,https://www.education.gov.za/LinkClick.aspx?fi...,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
3,2023_nov_p2_memo_maths,downloaded,https://www.education.gov.za/LinkClick.aspx?fi...,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
4,2024_nov_p1_exam_maths,downloaded,https://www.education.gov.za/Portals/0/CD/2024...,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
5,2024_nov_p1_memo_maths,downloaded,https://www.education.gov.za/LinkClick.aspx?fi...,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
6,2024_nov_p2_exam_maths,downloaded,https://www.education.gov.za/Portals/0/CD/2024...,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
7,2024_nov_p2_memo_maths,downloaded,https://www.education.gov.za/LinkClick.aspx?fi...,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
8,2025_nov_p1_exam_maths,downloaded,https://www.education.gov.za/LinkClick.aspx?fi...,c:\Users\Administrator\Desktop\Matric-Maths-Ex...
9,2025_nov_p1_memo_maths,downloaded,https://www.education.gov.za/LinkClick.aspx?fi...,c:\Users\Administrator\Desktop\Matric-Maths-Ex...


Successful: 14
Failed: 0


In [55]:
downloaded_df = register_df[register_df["collection_status"] == "downloaded"].copy()
validation_results = []

for _, row in downloaded_df.iterrows():
    file_path = PROJECT_ROOT / str(row["file_path"])
    exists = file_path.exists()
    size = file_path.stat().st_size if exists else 0
    is_pdf = False
    if exists:
        with open(file_path, "rb") as f:
            is_pdf = f.read(4) == b"%PDF"

    validation_results.append({
        "document_id": row["document_id"],
        "file_name": row["file_name"],
        "exists": exists,
        "size_bytes": size,
        "is_pdf": is_pdf,
        "valid": exists and is_pdf and size > 1000
    })

validation_df = pd.DataFrame(validation_results)
display(validation_df)

invalid = validation_df[~validation_df["valid"]]
if len(invalid):
    print("WARNING: invalid downloads")
    display(invalid)
else:
    print("All downloaded files valid")

,document_id,file_name,exists,size_bytes,is_pdf,valid
0,2023_nov_p1_exam_maths,NSC_Mathematics_2023_Nov_P1_exam.pdf,True,331213,True,True
1,2023_nov_p1_memo_maths,NSC_Mathematics_2023_Nov_P1_memo.pdf,True,692194,True,True
2,2023_nov_p2_exam_maths,NSC_Mathematics_2023_Nov_P2_exam.pdf,True,463246,True,True
3,2023_nov_p2_memo_maths,NSC_Mathematics_2023_Nov_P2_memo.pdf,True,790034,True,True
4,2024_nov_p1_exam_maths,NSC_Mathematics_2024_Nov_P1_exam.pdf,True,370676,True,True
5,2024_nov_p1_memo_maths,NSC_Mathematics_2024_Nov_P1_memo.pdf,True,527354,True,True
6,2024_nov_p2_exam_maths,NSC_Mathematics_2024_Nov_P2_exam.pdf,True,583436,True,True
7,2024_nov_p2_memo_maths,NSC_Mathematics_2024_Nov_P2_memo.pdf,True,731057,True,True
8,2025_nov_p1_exam_maths,NSC_Mathematics_2025_Nov_P1_exam.pdf,True,399263,True,True
9,2025_nov_p1_memo_maths,NSC_Mathematics_2025_Nov_P1_memo.pdf,True,615636,True,True


All downloaded files valid


In [56]:
register_df.to_csv(register_path, index=False)
check_df = pd.read_csv(register_path)

print("Updated register saved")
print(check_df["collection_status"].value_counts())

downloaded = check_df[check_df["collection_status"] == "downloaded"]
display(
    downloaded[
        ["document_id", "year", "paper", "document_type", "file_name", "file_path"]
    ]
)

NameError: name 'register_path' is not defined

In [57]:
# ============================================================
# FIX: DEFINE REGISTER PATH AND SAVE UPDATED REGISTER
# ============================================================

from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# Metadata directory
META_DIR = PROJECT_ROOT / "data" / "metadata"

# Document register
register_path = META_DIR / "exam_document_register.csv"

# Make sure the directory exists
META_DIR.mkdir(parents=True, exist_ok=True)

# Save updated register
register_df.to_csv(
    register_path,
    index=False
)

print("Updated register saved successfully.")
print()
print(f"Location: {register_path}")

Updated register saved successfully.

Location: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\metadata\exam_document_register.csv


In [58]:
# ============================================================
# RELOAD AND VERIFY REGISTER
# ============================================================

check_df = pd.read_csv(register_path)

print("Register successfully reloaded.")
print()

print("Collection status:")
print(check_df["collection_status"].value_counts())

print()
print(f"Total documents: {len(check_df)}")

Register successfully reloaded.

Collection status:
collection_status
queued        192
downloaded     14
Name: count, dtype: int64

Total documents: 206


In [59]:
# ============================================================
# FINAL DOWNLOAD VALIDATION
# ============================================================

downloaded = check_df[
    check_df["collection_status"] == "downloaded"
].copy()

print("=" * 60)
print("DOWNLOADED DOCUMENTS")
print("=" * 60)

for _, row in downloaded.iterrows():

    file_path = PROJECT_ROOT / row["file_path"]

    if file_path.exists():
        size_kb = file_path.stat().st_size / 1024

        print(
            f"✓ {row['file_name']} "
            f"({size_kb:.1f} KB)"
        )

    else:
        print(
            f"✗ MISSING: {row['file_name']}"
        )

print()
print(f"Total downloaded: {len(downloaded)}")

DOWNLOADED DOCUMENTS
✓ NSC_Mathematics_2023_Nov_P1_exam.pdf (323.5 KB)
✓ NSC_Mathematics_2023_Nov_P1_memo.pdf (676.0 KB)
✓ NSC_Mathematics_2023_Nov_P2_exam.pdf (452.4 KB)
✓ NSC_Mathematics_2023_Nov_P2_memo.pdf (771.5 KB)
✓ NSC_Mathematics_2024_Nov_P1_exam.pdf (362.0 KB)
✓ NSC_Mathematics_2024_Nov_P1_memo.pdf (515.0 KB)
✓ NSC_Mathematics_2024_Nov_P2_exam.pdf (569.8 KB)
✓ NSC_Mathematics_2024_Nov_P2_memo.pdf (713.9 KB)
✓ NSC_Mathematics_2025_Nov_P1_exam.pdf (389.9 KB)
✓ NSC_Mathematics_2025_Nov_P1_memo.pdf (601.2 KB)
✓ NSC_Mathematics_2025_Nov_P2_exam.pdf (461.5 KB)
✓ NSC_Mathematics_2025_Nov_P2_memo.pdf (389.9 KB)
✓ NSC_Mathematics_2011_caps.pdf (2896.9 KB)
✓ NSC_Mathematics_2021_guideline.pdf (502.7 KB)

Total downloaded: 14
